In [1]:
import json
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data")          # notebook lives in notebooks/, data is one level up
files = sorted(DATA_DIR.rglob("*.json"))
print(f"Found {len(files)} match files")

Found 3187 match files


In [4]:
with open(files[0], encoding="utf-8") as f:
    sample = json.load(f)

print(sample.keys())
print(json.dumps(sample["info"], indent=2)[:1500])
print(sample["innings"][0]["overs"][0]["deliveries"][0])

dict_keys(['meta', 'info', 'innings'])
{
  "balls_per_over": 6,
  "city": "Brisbane",
  "dates": [
    "2017-01-13"
  ],
  "event": {
    "match_number": 1,
    "name": "Pakistan in Australia ODI Series"
  },
  "gender": "male",
  "match_type": "ODI",
  "match_type_number": 3817,
  "officials": {
    "match_referees": [
      "JJ Crowe"
    ],
    "reserve_umpires": [
      "SJ Nogajski"
    ],
    "tv_umpires": [
      "CB Gaffaney"
    ],
    "umpires": [
      "MD Martell",
      "C Shamshuddin"
    ]
  },
  "outcome": {
    "by": {
      "runs": 92
    },
    "winner": "Australia"
  },
  "overs": 50,
  "player_of_match": [
    "MS Wade"
  ],
  "players": {
    "Australia": [
      "DA Warner",
      "TM Head",
      "SPD Smith",
      "CA Lynn",
      "MR Marsh",
      "GJ Maxwell",
      "MS Wade",
      "JP Faulkner",
      "PJ Cummins",
      "MA Starc",
      "B Stanlake"
    ],
    "Pakistan": [
      "Azhar Ali",
      "Sharjeel Khan",
      "Mohammad Hafeez",
      "Babar Az

In [6]:
#Parsing
def parse_match(path):
    with open(path, encoding="utf-8") as f:
        m = json.load(f)

    info = m["info"]
    match_id = path.stem                    # filename without .json is Cricsheet's match ID
    outcome = info.get("outcome", {})
    teams = info["teams"]

    match_row = {
        "match_id": match_id,
        "date": info["dates"][0],
        "gender": info.get("gender"),
        "team1": teams[0],
        "team2": teams[1],
        "venue": info.get("venue"),
        "city": info.get("city"),
        "toss_winner": info["toss"]["winner"],
        "toss_decision": info["toss"]["decision"],
        "winner": outcome.get("winner"),                  # None for ties / no result
        "result": outcome.get("result", "win" if "winner" in outcome else None),
        "method": outcome.get("method"),                  # e.g. "D/L" for rain-affected
        "eliminator": outcome.get("eliminator"),          # super-over winner after a tie
        "overs_limit": info.get("overs"),
        "target_runs": None,
    }

    ball_rows = []
    inn_no = 0
    for inn in m.get("innings", []):
        if inn.get("super_over"):
            continue
        inn_no += 1

        if inn_no == 2 and "target" in inn:               # official target (accounts for DLS)
            match_row["target_runs"] = inn["target"].get("runs")

        legal = 0
        for over in inn.get("overs", []):
            for d in over["deliveries"]:
                extras = d.get("extras", {})
                is_legal = ("wides" not in extras) and ("noballs" not in extras)
                legal += is_legal

                ball_rows.append({
                    "match_id": match_id,
                    "innings": inn_no,
                    "batting_team": inn["team"],
                    "over": over["over"],                 # 0-indexed: over 0 is the first over
                    "legal_ball_no": legal,               # legal balls bowled so far this innings
                    "is_legal": is_legal,
                    "batter": d["batter"],
                    "bowler": d["bowler"],
                    "runs_batter": d["runs"]["batter"],
                    "runs_extras": d["runs"]["extras"],
                    "runs_total": d["runs"]["total"],
                    "wicket": len(d.get("wickets", [])),  # usually 0 or 1
                })

    return match_row, ball_rows

In [8]:
matches, balls, errors = [], [], []

for p in files:
    try:
        match_row, ball_rows = parse_match(p)
        matches.append(match_row)
        balls.extend(ball_rows)
    except Exception as e:
        errors.append((p.name, repr(e)))

matches_df = pd.DataFrame(matches)
balls_df = pd.DataFrame(balls)

print(f"Parsed {len(matches_df)} matches, {len(balls_df):,} deliveries, {len(errors)} errors")
if errors:
    print(errors[:5])

Parsed 3187 matches, 1,686,437 deliveries, 0 errors


In [9]:
print(matches_df["gender"].value_counts())

matches_df = matches_df[matches_df["gender"] == "male"].reset_index(drop=True)
balls_df = balls_df[balls_df["match_id"].isin(matches_df["match_id"])].reset_index(drop=True)

OUT = DATA_DIR / "processed"
OUT.mkdir(exist_ok=True)
matches_df.to_csv(OUT / "matches.csv", index=False)
balls_df.to_csv(OUT / "balls.csv", index=False)
print("Saved:", matches_df.shape, balls_df.shape)

gender
male      2576
female     611
Name: count, dtype: int64
Saved: (2576, 15) (1366245, 12)


In [10]:
print(matches_df["date"].min(), "to", matches_df["date"].max())
print(matches_df["result"].value_counts(dropna=False))
print(matches_df["method"].value_counts(dropna=False))
print(balls_df.groupby(["match_id", "innings"])["legal_ball_no"].max().describe())

2002-06-27 to 2026-09-15
result
win          2447
no result     102
tie            27
Name: count, dtype: int64
method
NaN    2354
D/L     222
Name: count, dtype: int64
count    5084.000000
mean      261.488788
std        55.639862
min         4.000000
25%       241.000000
50%       289.000000
75%       300.000000
max       301.000000
Name: legal_ball_no, dtype: float64
